# NOVA RETAIL — Auditoría Inicial de Datos

## Evaluación Forense de Calidad de Datos
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Recepción inicial y revisión de integridad de datos  
**Objetivo:** Evaluar la estructura, calidad y señales iniciales de riesgo en los datos crudos recibidos de NOVA RETAIL.

---

### Alcance de este notebook
Este notebook documenta la primera auditoría forense sobre los datos crudos entregados para el caso NOVA RETAIL. El propósito es:

- verificar disponibilidad y estructura de archivos
- evaluar problemas de calidad de datos en los datasets principales
- identificar riesgos inmediatos de integridad
- establecer una línea base para la reconciliación y detección de anomalías posterior

### Datasets crudos esperados
- `stores.csv`
- `products_sap.csv`
- `products_as400.csv`
- `employees.csv`
- `pos_transactions.csv`

### Nota del analista
Este notebook no busca concluir fraude.  
Este es un notebook de **diagnóstico de integridad de datos y operación**, diseñado para separar señal de ruido antes de cualquier escalamiento investigativo.

In [ ]:
%pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

print("Notebook listo.")
print("Ruta de datos:", DATA_PATH.resolve())

In [ ]:
stores = pd.read_csv(DATA_PATH / "stores.csv")
products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")
employees = pd.read_csv(DATA_PATH / "employees.csv")
pos = pd.read_csv(DATA_PATH / "pos_transactions.csv")

print("Datasets cargados correctamente.")

In [ ]:
datasets = {
    "stores": stores,
    "products_sap": products_sap,
    "products_as400": products_as400,
    "employees": employees,
    "pos_transactions": pos
}

summary = pd.DataFrame({
    "dataset": datasets.keys(),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

summary

In [ ]:
datasets = {
    "stores": stores,
    "products_sap": products_sap,
    "products_as400": products_as400,
    "employees": employees,
    "pos_transactions": pos
}

summary = pd.DataFrame({
    "dataset": list(datasets.keys()),
    "filas": [df.shape[0] for df in datasets.values()],
    "columnas": [df.shape[1] for df in datasets.values()],
    "memoria_mb": [df.memory_usage(deep=True).sum() / 1024**2 for df in datasets.values()]
})

summary.style.format({
    "filas": "{:,}",
    "memoria_mb": "{:.1f}"
}).background_gradient(cmap='viridis')

In [ ]:
# Distribución de tiendas por sistema
fig = px.pie(
    values=stores['system'].value_counts(),
    names=stores['system'].value_counts().index,
    title="Distribución de tiendas por sistema",
    color_discrete_map={'SAP': '#1f77b4', 'AS400': '#ff7f0e'}
)

fig.show()

In [ ]:
# Porcentaje de valores nulos por dataset
null_summary = []

for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    total_nulls = df.isna().sum().sum()
    pct_nulls = (total_nulls / total_cells) * 100 if total_cells > 0 else 0

    null_summary.append({
        "dataset": name,
        "total_celdas": total_cells,
        "valores_nulos": total_nulls,
        "porcentaje_nulos": pct_nulls
    })

null_summary_df = pd.DataFrame(null_summary)
null_summary_df.style.format({
    "total_celdas": "{:,}",
    "valores_nulos": "{:,}",
    "porcentaje_nulos": "{:.2f}%"
}).background_gradient(subset=["porcentaje_nulos"], cmap="Reds")

In [ ]:
ghost_stores = stores[stores["ghost_store"] == True]

print(f"Total de ghost stores: {len(ghost_stores)}")
ghost_stores[[
    "store_id", "store_name", "city", "state", "system",
    "cedis_route", "regional_manager_id"
]].sort_values("store_id")

In [ ]:
as400_quality = {
    "total_registros": len(products_as400),
    "sku_as400_nulos": products_as400["sku_as400"].isna().sum(),
    "description_as400_nulos": products_as400["description_as400"].isna().sum(),
    "category_as400_vacias": (products_as400["category_as400"] == "").sum(),
    "price_as400_nulos": products_as400["price_as400"].isna().sum(),
    "apple_ghosts": products_as400["is_ghost"].sum()
}

pd.DataFrame(as400_quality.items(), columns=["metrica", "valor"])

In [ ]:
pos_nulls = (
    pos.isna()
    .sum()
    .reset_index()
)

pos_nulls.columns = ["columna", "nulos"]
pos_nulls["porcentaje"] = (pos_nulls["nulos"] / len(pos)) * 100
pos_nulls = pos_nulls.sort_values("nulos", ascending=False)

pos_nulls.style.format({
    "nulos": "{:,}",
    "porcentaje": "{:.2f}%"
}).background_gradient(subset=["porcentaje"], cmap="Reds")

In [ ]:
as400_nulls = (
    products_as400.isna()
    .sum()
    .reset_index()
)

as400_nulls.columns = ["columna", "nulos"]
as400_nulls["porcentaje"] = (as400_nulls["nulos"] / len(products_as400)) * 100
as400_nulls = as400_nulls.sort_values("nulos", ascending=False)

as400_nulls.style.format({
    "nulos": "{:,}",
    "porcentaje": "{:.2f}%"
}).background_gradient(subset=["porcentaje"], cmap="Oranges")

In [ ]:
# Conversión de timestamp sin intentar renderizar demasiado
pos["timestamp"] = pd.to_datetime(pos["timestamp"], errors="coerce")

year_counts = pos["timestamp"].dt.year.value_counts(dropna=False).sort_index()
print(year_counts)

In [ ]:
timestamps_1970_sample = pos.loc[
    pos["timestamp"].dt.year == 1970,
    ["transaction_id", "store_id", "sku", "timestamp", "system_source", "sync_status"]
].head(10)

timestamps_1970_sample

In [ ]:
pos_nulls = (
    pos.isna()
    .sum()
    .reset_index()
)

pos_nulls.columns = ["columna", "nulos"]
pos_nulls["porcentaje"] = (pos_nulls["nulos"] / len(pos)) * 100
pos_nulls = pos_nulls.sort_values("nulos", ascending=False)

pos_nulls

In [ ]:
as400_nulls = (
    products_as400.isna()
    .sum()
    .reset_index()
)

as400_nulls.columns = ["columna", "nulos"]
as400_nulls["porcentaje"] = (as400_nulls["nulos"] / len(products_as400)) * 100
as400_nulls = as400_nulls.sort_values("nulos", ascending=False)

as400_nulls